##### This notebook's goal:
To preprocess, merge, and otherwise prepare the original datasets (JSON) while converting them to CSVs. This includes some text normalisation, tokenisation, and feature engineering steps.

##### Author(s):
- (Ahn) Michael Howell - Human Language Technology Masters - Sociolinguist & Agentic AI Engineer - ahn@equita-tech.com

#### Gather imports

In [1]:
import numpy as np
import pandas as pd
import re # For text processing, pattern recognition
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS # For focusing NLP analysis on contentful words

#### Load original data

In [2]:
## LOAD DATA
reddit_url = "https://raw.githubusercontent.com/taivop/joke-dataset/refs/heads/master/reddit_jokes.json"
stupid_stuff_url = "https://raw.githubusercontent.com/taivop/joke-dataset/refs/heads/master/stupidstuff.json"
wocka_url = "https://raw.githubusercontent.com/taivop/joke-dataset/refs/heads/master/wocka.json"

reddit_df = pd.read_json(reddit_url) # Extra columns: title, score
ss_df = pd.read_json(stupid_stuff_url) # Extra columns: rating, category
wocka_df = pd.read_json(wocka_url) # Extra columns: title, category

##### Preprocessing: tidy initial columns & merging

In [ ]:
# Add site column to each
reddit_df["website"] = "reddit"
ss_df["website"] = "stupidstuff"
wocka_df["website"] = "wocka"

# Ensure each dataset has all necessary columns
for d in (reddit_df, ss_df, wocka_df):
    for col in ['title', 'category', 'score', 'rating']:
        if col not in d.columns:
            d[col] = pd.NA

## MERGE DATASETS
merged_df = pd.concat([reddit_df, ss_df, wocka_df], ignore_index=True)

# Ensure text columns are real strings (empty if missing)
merged_df['title'] = merged_df['title'].fillna('')
merged_df['body'] = merged_df['body'].fillna('')
merged_df['category'] = merged_df['category'].fillna('')

# Ensure score is numeric
merged_df['score'] = pd.to_numeric(merged_df['score'], errors='coerce')

## REVIEW UPDATES
def show_df_details(details_to_include = []):
    if "head" in details_to_include:
        print(f"\n** merged_df.head:\n{merged_df.head()}\n")
    if "tail" in details_to_include:
        print(f"\n** merged_df.tail:\n{merged_df.tail()}\n")
    if "info" in details_to_include:
        merged_df.info()
        # print(f"\n** merged_df.info:\n{merged_df.info(verbose = 'true')}\n")
    if "describe" in details_to_include:
        print(f"\n** merged_df.describe:\n{merged_df.describe()}\n")
    if "describe-all" in details_to_include:
        print(f"\n** merged_df.describe:\n{merged_df.describe(include = 'all')}\n")

show_df_details(["head", "tail", "info", "describe-all"])
# Consider if rating & score can be immediately merged > no, they are not on the same scale
# Rating is 0 to 5, while score is upvotes count; TODO: reflect on binning strategy
# to fit into 5-part scale: VERYLOW, LOW, MID, HIGH, VERYHIGH, as user_rating column
# TODO: look at distribution of score column, review outliers
# TODO: reflect on if the 5-part binning makes more sense or keeping numerics; maybe both
# Score is extremely skewed (median = 3, max = 48,526, 75% quantile = 16, means 75% of jokes have 16 upvotes or less)
# Standard deviation = 936; variance is mostly caused by viral jokes > should use quantiles to bin
print(f"\nscore & rating .describe:\n{merged_df[['score','rating']].describe(include='all')}\n")
# TODO: check on if the IDs will be a challenge > it may make most sense to just keep reddit data only

## TEMPORARY FILTER - to simplify data intake
websites_to_keep = ["reddit"]
columns_to_keep = ["body", "id", "title", "score", "website"]
merged_df = merged_df.loc[merged_df['website'].isin(websites_to_keep), columns_to_keep].copy()
print(f"\n** AFTER FILTER: merged_df.head:\n{merged_df.head()}\n")

/tmp/ipykernel_61142/2405821299.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_df = pd.concat([reddit_df, ss_df, wocka_df], ignore_index=True)



** merged_df.head:
                                                body      id  score  \
0  Now I have to say "Leroy can you please paint ...  5tz52q    1.0   
1  Pizza doesn't scream when you put it in the ov...  5tz4dd    0.0   
2  ...and being there really helped me learn abou...  5tz319    0.0   
3  A Sunday school teacher is concerned that his ...  5tz2wj    1.0   
4  He got caught trying to sell the two books to ...  5tz1pc    0.0   

                                               title website category  rating  
0   I hate how you cant even say black paint anymore  reddit              NaN  
1  What's the difference between a Jew in Nazi Ge...  reddit              NaN  
2                     I recently went to America....  reddit              NaN  
3  Brian raises his hand and says, “He’s in Heaven.”  reddit              NaN  
4  You hear about the University book store worke...  reddit              NaN  


** merged_df.tail:
                                                    

##### Preprocessing: gather whole text & initial text features

In [4]:
# Add feature: whole_text (string) - the first and second parts of the jokes combined
merged_df['whole_text'] = (merged_df['title'] + ' ' + merged_df['body']).str.strip()

# Add feature: whole_text_normalised (string) - to support further NLP analysis
def normalise_text(s):
    s = s.lower() # make lowercase
    s = re.sub(r"[^a-z0-9\s]", " ", s) # remove non-text characters
    s = re.sub(r"\s+", " ", s)  # collapse multiple spaces to a single space
    s = s.strip() # remove whitespace from ends
    return s
merged_df['whole_text_normalised'] = merged_df['whole_text'].apply(normalise_text)

# Add feature: whole_text_normalised_tokens (list) - to support further NLP analysis
merged_df['whole_text_normalised_tokens'] = merged_df['whole_text_normalised'].str.split()

# Add feature: overall_length (int) - count of the words in the text (not normalised, stop words remain)
merged_df['overall_length'] = merged_df['whole_text_normalised_tokens'].apply(len)

# Add feature: distinct_words (list) - the list of different words in the text (stop words removed)
CONTRACTED_STOP_WORDS = {"im","ive","youre","dont","didnt","hes","shes","theyre","weve","cant","wont","thats"}
STOP_WORDS_ALL = set(ENGLISH_STOP_WORDS) | CONTRACTED_STOP_WORDS
merged_df['distinct_words'] = merged_df['whole_text_normalised_tokens'].apply(
    lambda toks: list(set(t for t in toks if t not in STOP_WORDS_ALL))
)

# Add feature: distinct_words_count (int) - the number of different words used
merged_df['distinct_words_count'] = merged_df['distinct_words'].apply(len)

## REVIEW UPDATES
show_df_details(["head", "tail", "info", "describe"])


** merged_df.head:
                                                body      id  \
0  Now I have to say "Leroy can you please paint ...  5tz52q   
1  Pizza doesn't scream when you put it in the ov...  5tz4dd   
2  ...and being there really helped me learn abou...  5tz319   
3  A Sunday school teacher is concerned that his ...  5tz2wj   
4  He got caught trying to sell the two books to ...  5tz1pc   

                                               title  score website  \
0   I hate how you cant even say black paint anymore    1.0  reddit   
1  What's the difference between a Jew in Nazi Ge...    0.0  reddit   
2                     I recently went to America....    0.0  reddit   
3  Brian raises his hand and says, “He’s in Heaven.”    1.0  reddit   
4  You hear about the University book store worke...    0.0  reddit   

                                          whole_text  \
0  I hate how you cant even say black paint anymo...   
1  What's the difference between a Jew in Nazi Ge...   


##### Preprocessing: gather theme counts from features

##### Tidy dataframe - last steps

#### Output new CSV (in data folder)